# Reasoning Language Model (RLM)
Chain-of-thought (Wei et al., 2022), scratchpads (Nye et al., 2021), OpenAI o1 (2024), DeepSeek-R1 / GRPO (Shao et al., 2024; DeepSeek-AI, 2025)

**Idea:** let the model write intermediate steps ("think") before the answer. Each step is an easy local computation, so a problem the model can't solve in one forward pass becomes solvable. Reasoning models are then trained with **reinforcement learning** on problems with checkable answers: sample several solutions, reward the correct ones, and push the model toward what worked.

**Toy task:** sum of 8 digits mod 10. `37295184=` -> `9`.
- Direct answer: the model must combine all 8 digits in one pass (hard for a small Transformer).
- With reasoning: write the running sum after each digit, `3 0 2 1 6 7 5 9`, then the answer. Each step only needs the previous sum and one digit.

### RLM vs SLM
| | SLM (`slm.ipynb`) | RLM |
|---|---|---|
| Architecture | decoder-only Transformer | **same** decoder-only Transformer |
| Output | answer directly | reasoning tokens, then answer |
| Compute per question | fixed ($\propto$ answer length) | grows with the length of the reasoning ("test-time compute") |
| Training | next-token prediction (SFT) | SFT warm start, then **RL with verifiable rewards** (GRPO) |
| Reward | - | 1 if the final answer is correct, else 0 (only the answer is checked, not the steps) |

Shape flow: `prompt (B, 9) -> generate reasoning + answer (B, 11) -> reward (B,) from the answer token`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - generated on the fly
Infinite fresh problems, so there is no train/test split to overfit.

Vocab: `_` (unused pad), digits `0-9`, `=` (end of prompt), `>` (end of reasoning), `$` (end of answer). Token id of digit $d$ is $d + 1$.

| mode | prompt | target |
|---|---|---|
| direct | `37295184=` | `9$` |
| CoT (reasoning) | `37295184=` | `30216759>9$` |

In [ ]:
chars = "_0123456789=>$"
stoi = {c: i for i, c in enumerate(chars)}
EQ, GT, END = stoi["="], stoi[">"], stoi["$"]
vocab_size = len(chars)
n_digits = 8

def make_batch(B, cot):
    # -> prompt (B, 9), target (B, 2) direct or (B, 11) cot, answer (B,)
    d = torch.randint(0, 10, (B, n_digits), device=device)    # (B, 8) digits
    run = d.cumsum(dim=1) % 10                                 # (B, 8) running sums mod 10 = the reasoning steps
    ans = run[:, -1]                                           # (B,)
    col = lambda tok: torch.full((B, 1), tok, device=device)
    prompt = torch.cat([d + 1, col(EQ)], dim=1)
    if cot:
        target = torch.cat([run + 1, col(GT), ans[:, None] + 1, col(END)], dim=1)
    else:
        target = torch.cat([ans[:, None] + 1, col(END)], dim=1)
    return prompt, target, ans

decode = lambda ids: "".join(chars[i] for i in ids)
p, t, a = make_batch(2, cot=True)
for i in range(2):
    print(decode(p[i].tolist()), "->", decode(t[i].tolist()))

## 2. Model - a small GPT
Standard decoder: token + learned position embeddings, Pre-LN blocks with causal attention (the ViT block plus a mask), LayerNorm, linear head. Nothing reasoning-specific: **reasoning is a property of the data and training, not of the architecture.**

### Formula: Causal attention
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V, \qquad M_{mn} = -\infty \text{ for } n > m$$
- $Q, K, V \in \mathbb{R}^{T \times d_k}$ per head, $d_k = 32$
- $M$: causal mask; each position sees only itself and earlier tokens, so it can read the reasoning written so far

In [ ]:
class CausalSelfAttention(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads, self.head_dim = num_heads, dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)  # each (B, h, T, hd)
        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5                   # (B, h, T, T)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
        att = att.masked_fill(~mask, float("-inf")).softmax(dim=-1)
        return self.proj((att @ v).transpose(1, 2).reshape(B, T, D))             # (B, T, D)


class Block(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, num_heads)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # (B, T, D)
        x = x + self.mlp(self.ln2(x))   # (B, T, D)
        return x


class GPT(nn.Module):
    # idx (B, T) -> logits (B, T, V)
    def __init__(self, vocab_size, max_len=32, dim=128, depth=4, num_heads=4):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, dim))
        self.blocks = nn.Sequential(*[Block(dim, num_heads) for _ in range(depth)])
        self.ln = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, idx):
        x = self.tok_embed(idx) + self.pos_embed[:, :idx.shape[1]]  # (B, T, D)
        return self.head(self.ln(self.blocks(x)))                   # (B, T, V)

print(f"params: {sum(p.numel() for p in GPT(vocab_size).parameters()) / 1e6:.2f}M")

## 3. Supervised training (SFT) and evaluation
Train on `prompt + target`, but only count the loss on the target tokens: the prompt is random digits, there is nothing to learn from predicting it.

Accuracy = greedy-decode the target, read the answer token, compare to the true answer. Random guessing scores 10%.

### Formula: SFT loss (masked next-token cross-entropy)
$$\mathcal{L}_{SFT} = -\frac{1}{|y|}\sum_{t=1}^{|y|} \log \pi_\theta(y_t \mid x, y_{<t})$$
- $x$: prompt (`37295184=`), $y$: target (`30216759>9$` or `9$`)
- $\pi_\theta$: the model's next-token distribution (we call it the **policy** in the RL section)
- positions inside the prompt get label `-100` (ignored by `F.cross_entropy`)

### Formula: Why reasoning helps
$$p(a \mid x) = \sum_{z} p(z \mid x)\, p(a \mid x, z)$$
- $z$: the reasoning (running sums), $a$: the answer
- direct: one forward pass must compute $a$ from all 8 digits at once
- CoT: each token of $z$ is one easy step $z_i = (z_{i-1} + x_i) \bmod 10$, and $a$ is a copy of $z_8$; more tokens = more sequential compute

In [ ]:
def sft_step(model, optimizer, cot, batch_size=256):
    prompt, target, _ = make_batch(batch_size, cot)
    seq = torch.cat([prompt, target], dim=1)                  # (B, 9 + len(target))
    x, y = seq[:, :-1], seq[:, 1:].clone()                    # predict the next token
    y[:, :n_digits] = -100                                    # no loss on the prompt
    logits = model(x)                                         # (B, T, V)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1), ignore_index=-100)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def generate(model, prompt, n_new, temperature=0.0):
    # prompt (B, 9) -> generated tokens (B, n_new); temperature 0 = greedy
    seq = prompt
    for _ in range(n_new):
        logits = model(seq)[:, -1]                                            # (B, V)
        if temperature == 0:
            next_id = logits.argmax(-1, keepdim=True)                         # (B, 1)
        else:
            next_id = torch.multinomial(F.softmax(logits / temperature, -1), 1)
        seq = torch.cat([seq, next_id], dim=1)
    return seq[:, prompt.shape[1]:]

def answer_of(gen, cot):
    return (gen[:, n_digits + 1] if cot else gen[:, 0]) - 1                  # answer token -> digit (B,)

@torch.no_grad()
def accuracy(model, cot, n=2000):
    model.eval()
    prompt, target, ans = make_batch(n, cot)
    acc = (answer_of(generate(model, prompt, target.shape[1]), cot) == ans).float().mean().item()
    model.train()
    return acc

### 3a. Direct answer

In [ ]:
direct = GPT(vocab_size).to(device)
opt = torch.optim.AdamW(direct.parameters(), lr=1e-3)
for it in range(1, 2001):
    loss = sft_step(direct, opt, cot=False)
    if it % 500 == 0:
        print(f"direct | step {it:4d} | loss {loss:.3f} | acc {accuracy(direct, cot=False):.3f}")

### 3b. With reasoning (CoT) - short warm start
Same model, same optimizer, targets now include the running sums. We **stop early on purpose** once greedy accuracy passes 40%, to leave room for RL in the next section. (Keep training and SFT alone reaches ~100% within a few hundred more steps.)

In [ ]:
cot_model = GPT(vocab_size).to(device)
opt = torch.optim.AdamW(cot_model.parameters(), lr=1e-3)
it, acc = 0, 0.0
while acc < 0.4:
    it += 1
    loss = sft_step(cot_model, opt, cot=True)
    if it % 10 == 0:
        acc = accuracy(cot_model, cot=True, n=500)
    if it % 50 == 0:
        print(f"cot | step {it:4d} | loss {loss:.3f} | acc {acc:.3f}")
print(f"stopped at step {it}, acc {accuracy(cot_model, cot=True):.3f}")

prompt, target, ans = make_batch(3, cot=True)
for p, g in zip(prompt, generate(cot_model, prompt, target.shape[1])):
    print(decode(p.tolist()), "->", decode(g.tolist()))

## 4. Reinforcement learning with GRPO
Group Relative Policy Optimization (DeepSeekMath, 2024), the algorithm behind DeepSeek-R1. For each prompt, sample a **group** of $G$ answers, score them with a verifiable reward, and use each sample's reward relative to its group as its advantage. No value network, no labels for the reasoning steps: the model has to find reasoning that leads to correct answers.

### Formula: Reward
$$r_i = \mathbb{1}\left[\text{answer}(o_i) = a^*\right]$$
- $o_i$: the $i$-th sampled output (reasoning + answer) for prompt $q$
- $a^*$: the true answer; only the final answer is checked

### Formula: Group-relative advantage
$$\hat A_i = \frac{r_i - \text{mean}(r_1, \dots, r_G)}{\text{std}(r_1, \dots, r_G) + \epsilon}$$
- $G = 8$ samples per prompt
- correct samples in a mostly-wrong group get a large positive advantage; wrong ones slightly negative
- if all $G$ samples score the same, every $\hat A_i = 0$: no signal (why the SFT warm start is needed)
- $\epsilon$: $10^{-4}$, avoids division by zero

### Formula: Policy-gradient loss
$$\mathcal{L}_{GRPO} = -\frac{1}{G}\sum_{i=1}^{G} \frac{1}{|o_i|}\sum_{t=1}^{|o_i|} \hat A_i \log \pi_\theta(o_{i,t} \mid q, o_{i,<t})$$
- $\pi_\theta(o_{i,t} \mid \cdot)$: probability of the token the model actually sampled
- raises the log-prob of every token in above-average samples, lowers it for below-average ones
- full GRPO also has a PPO-style ratio clip and a KL penalty to a reference model; with one gradient step per fresh batch the ratio is 1, so this is the same update without those extras
- small learning rate (3e-5): RL updates are noisy and a large step destroys the SFT model

In [ ]:
G, P = 8, 32                      # samples per prompt, prompts per step
rl_steps = 300
opt = torch.optim.AdamW(cot_model.parameters(), lr=3e-5)

for step in range(1, rl_steps + 1):
    prompt, target, ans = make_batch(P, cot=True)
    prompt, ans = prompt.repeat_interleave(G, 0), ans.repeat_interleave(G, 0)  # (P*G, 9), (P*G,)
    n_new = target.shape[1]

    cot_model.eval()
    with torch.no_grad():
        out = generate(cot_model, prompt, n_new, temperature=1.0)             # (P*G, 11) sampled outputs
    cot_model.train()

    reward = (answer_of(out, cot=True) == ans).float()                        # (P*G,)
    r = reward.view(P, G)
    adv = ((r - r.mean(1, keepdim=True)) / (r.std(1, keepdim=True) + 1e-4)).view(-1)  # (P*G,)

    seq = torch.cat([prompt, out], dim=1)                                     # (P*G, 20)
    logits = cot_model(seq[:, :-1])[:, prompt.shape[1] - 1:]                  # positions that predicted `out` -> (P*G, 11, V)
    logp = F.log_softmax(logits, dim=-1).gather(-1, out[..., None])[..., 0]   # (P*G, 11)
    loss = -(adv[:, None] * logp).mean()

    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(f"grpo | step {step:3d} | mean reward (sampled) {reward.mean():.3f} | greedy acc {accuracy(cot_model, cot=True):.3f}")

## Notes
- Expected pattern: the direct model stays at ~10% (chance), the CoT warm start stops at ~40%, and GRPO pushes it to ~99% using only a right/wrong signal on the final answer.
- Real RLMs (o1, R1, Qwen-QwQ) are regular LLMs trained this way on math and code with automatic checkers. Long "thinking" traces, self-checking and backtracking emerge from the RL because they raise the reward.
- The same idea at inference time: more reasoning tokens (or more samples plus majority vote) = more **test-time compute** = better answers on hard problems.
- **LAM** (next) keeps the "generate, then get feedback" loop, but its outputs are **actions** in an environment instead of text.